In [13]:
from typing_extensions import TypedDict, Literal
from typing import List
from langgraph.types import Command
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-4o")

dumb_llm = init_chat_model("openai:gpt-3.5-turbo")
average_llm = init_chat_model("openai:gpt-4o")
smart_llm = init_chat_model("openai:gpt-5-mini-2025-08-07")

In [14]:
class State(TypedDict):
    question: str
    difficulty: list[dict]
    answer: str
    model_used: str

class DifficultyResponse(BaseModel):
    difficulty_level : Literal["easy", "medium", "hard"]


In [19]:
def dumb_node(state: State):
    response = dumb_llm.invoke(state["question"])
    return {
        "answer": response.content,
        "model_used": "gpt-3.5"
    }

def average_node(state: State):
    response = average_llm.invoke(state["question"])
    return {
        "answer": response.content,
        "model_used": "gpt-4o"
    }

def smart_node(state: State):
    response = smart_llm.invoke(state["question"])
    return {
        "answer": response.content,
        "model_used": "gpt-5-mini"
    }

def assess_difficulty(state: State): 
    structured_llm = llm.with_structured_output(DifficultyResponse)
    response = structured_llm.invoke(
    f"""
    이 질문의 난이도를 평가해 
    질문: {state["question"]}

    - 쉬움: 간단한 사실, 기본정의, 예/아니오 대답에 해당
    - 중간: 설명, 비교 분석이 필요해
    - 어려움: 여러 단계와 깊은 전문성이 필요해
    """)

    difficulty_level = response.difficulty_level

    if difficulty_level == "easy":
        goto = "dumb_node"
    elif difficulty_level == "medium":
        goto = "average_node"
    elif difficulty_level == "hard":
        goto = "smart_node"
    
    return Command(goto=goto, update={"difficulty": difficulty_level})

In [20]:
graph_builder = StateGraph(State)

graph_builder.add_node("dumb_node", dumb_node)
graph_builder.add_node("average_node", average_node)
graph_builder.add_node("smart_node", smart_node)
graph_builder.add_node("assess_difficulty", assess_difficulty, destinations=("dumb_node", "average_node", "smart_node"))

graph_builder.add_edge(START, "assess_difficulty")
graph_builder.add_edge("dumb_node", END)
graph_builder.add_edge("average_node", END)
graph_builder.add_edge("smart_node", END)

graph = graph_builder.compile()

In [22]:
graph.invoke({"question": "한국의 미래 투자에 대한 추천할만한 종목 알려줘"})


{'question': '한국의 미래 투자에 대한 추천할만한 종목 알려줘',
 'difficulty': 'hard',
 'answer': '좋습니다. 먼저 몇 가지 확인하면 더 적합한 종목을 추천해 드릴 수 있습니다.\n- 투자 기간(단기/중기(3~5년)/장기(5년 이상))  \n- 위험 성향(안정형/중립/공격형)  \n- 개별 종목 선호 vs ETF·펀드 같은 분산투자 선호 여부\n\n우선 한국의 미래 성장 테마별로 추천할 만한 업종과 대표 종목(혹은 ETF)을 요약해 드립니다. 각 항목에 간단한 투자 포인트와 주요 리스크도 같이 적었습니다.\n\n1) 반도체(메모리·파운드리·장비)\n- 대표 종목: 삼성전자, SK하이닉스 / 반도체 장비주: 원익IPS, 테스, 한미반도체 등\n- 투자 포인트: AI·데이터센터 수요, 고용량 메모리·파운드리 확장.\n- 리스크: 경기 민감, 가격 사이클/재고 리스크, 기술 경쟁 심화.\n\n2) 2차전지·전기차 배터리·소재\n- 대표 종목: LG에너지솔루션, 삼성SDI, LG화학(배터리 소재·화학), POSCO(배터리 소재·양극재 전환) / 관련 ETF: TIGER KRX BBIG, TIGER 전기차 등\n- 투자 포인트: EV 확대·배터리 수요 장기 성장, 배터리 소재 업체 가치 상승 가능성.\n- 리스크: 원자재 가격, 기술 대체, 고객(완성차) 경쟁·수요 변동.\n\n3) 바이오·CDMO(위탁생산)\n- 대표 종목: 삼성바이오로직스, 셀트리온\n- 투자 포인트: 바이오 의약품 수탁생산(CDMO) 시장 확대, 고마진 서비스.\n- 리스크: 임상 실패·규제 리스크, 계약·수주 변동성.\n\n4) 인터넷·플랫폼·AI\n- 대표 종목: 네이버, 카카오\n- 투자 포인트: 광고·커머스·핀테크·AI 서비스 확대, 플랫폼의 네트워크 효과.\n- 리스크: 규제(플랫폼·개인정보), 경쟁 심화, 수익 모델 전환 실패 가능성.\n\n5) 친환경·신재생에너지\n- 대표 종목: 한화솔루션(태양광), 두산퓨얼셀(수소연료전지 관련), 한화에너지\